In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

### **Data Reading**

In [0]:
df=spark.read.format('delta')\
    .load('abfss://bronze@adventwork.dfs.core.windows.net/productSubcategories')

### **Schema**

In [0]:
df.printSchema()

### **Changing ProductSubcategoryKey columns' format**

In [0]:
df=df.withColumn('ProductSubcategoryKey',col('ProductSubcategoryKey').cast('int'))

### **Changing ProductCategoryKey columns' format**

In [0]:
df=df.withColumn('ProductCategoryKey',col('ProductCategoryKey').cast('int'))

### **Deleting _rescued_data column**

In [0]:
df=df.drop("_rescued_data")

### **Fixing SubcategoryName column's data quality**

In [0]:
df=df.withColumn('SubcategoryName', trim(col('SubcategoryName')))

### **Data Writing**

In [0]:
if spark.catalog.tableExists('adventure_works.silver.productSubcategories'):
    df_silver_subcategory = spark.read.table('adventure_works.silver.productSubcategories')
    df=df.join(df_silver_subcategory,'ProductSubcategoryKey','left_anti')


In [0]:
df.write.format('delta').mode('append')\
    .save('abfss://silver@adventwork.dfs.core.windows.net/productSubcategories')

In [0]:
%sql
create table if not exists adventure_works.silver.productSubcategories
using delta
location 'abfss://silver@adventwork.dfs.core.windows.net/productSubcategories' 
